In [15]:
# =============================================================================
# GUS01F: Loading Numerical Data onto GeoTERYT Records
# =============================================================================
# This notebook demonstrates the v4.0 data storage capabilities:
# 1. Load BDL demographic data (subject P2137 - population)
# 2. Process and attach time series to TERYTRecord objects
# 3. Query data on individual records
# 4. Aggregate for regions (voivodeships)
# 5. Produce joint/marginal distributions
# 6. Save/reload database with data persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

# Reload geoTERYT_db to get latest version (v4.0)
import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"Data root: {data_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
Data root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [16]:
# =============================================================================
# STEP 2: Load Complete GeoTERYT Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [17]:
# =============================================================================
# STEP 3: Load BDL Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_c_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')

df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / "metadata" / 'census_meta.csv', encoding='utf-8')

print(f"df_demographic: {df_demographic.shape}")
print(f"df_variables: {df_variables.shape}")
print(f"\nAvailable subjects: {sorted(df_demographic['subjectId'].unique())}")

df_demographic: (431358, 5)
df_variables: (27922, 10)

Available subjects: ['P1336', 'P2137', 'P2914']


In [18]:
# Edit years for census of 1988

for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values']= df_c_1988.at[id, 'values'].replace(", 'year': '1998'", ", 'year': '1988'")
    
df_c_variables['years'] = df_c_variables['years'].str.replace("[1998]", "[1988]")

# Unify the meta dataframe structure
df_c_variables['n4'] = None
df_c_variables['n5'] = None


In [22]:
display(df_c_1988)
display(df_demographic)

,id,name,values,variableId,subjectId
0,11212001011,Bochnia,"[{'attrId': 1, 'val': 28379, 'year': '1988'}]",196133,P2884
1,11212001022,Bochnia,"[{'attrId': 1, 'val': 15384, 'year': '1988'}]",196133,P2884
2,11212001032,Drwinia,"[{'attrId': 1, 'val': 6099, 'year': '1988'}]",196133,P2884
3,11212001042,Lipnica Murowana,"[{'attrId': 1, 'val': 5017, 'year': '1988'}]",196133,P2884
4,11212001052,Łapanów,"[{'attrId': 1, 'val': 6819, 'year': '1988'}]",196133,P2884
...,...,...,...,...,...
68946,71427338024,Mszczonów - miasto,"[{'attrId': 1, 'val': 959, 'year': '1988'}]",196152,P2887
68947,71427338025,Mszczonów - obszar wiejski,"[{'attrId': 1, 'val': 531, 'year': '1988'}]",196152,P2887
68948,71427338032,Puszcza Mariańska,"[{'attrId': 1, 'val': 1071, 'year': '1988'}]",196152,P2887
68949,71427338042,Radziejowice,"[{'attrId': 1, 'val': 579, 'year': '1988'}]",196152,P2887


,id,name,values,variableId,subjectId
0,0,POLSKA,"[{'year': '1995', 'val': 38587596, 'attrId': 1...",60616,P1336
1,10000000000,MAKROREGION POŁUDNIOWY,"[{'year': '1995', 'val': 8098144, 'attrId': 1}...",60616,P1336
2,11200000000,MAŁOPOLSKIE,"[{'year': '1995', 'val': 3183199, 'attrId': 1}...",60616,P1336
3,11210000000,REGION MAŁOPOLSKIE,"[{'year': '1995', 'val': 3183199, 'attrId': 1}...",60616,P1336
4,11212000000,PODREGION KRAKOWSKI,"[{'year': '1995', 'val': 630080, 'attrId': 1},...",60616,P1336
...,...,...,...,...,...
431353,71427338042,Radziejowice,"[{'year': '1995', 'val': 0, 'attrId': 0}, {'ye...",199191,P2914
431354,71427338052,Wiskitki,"[{'year': '1995', 'val': 0, 'attrId': 0}, {'ye...",199191,P2914
431355,71427338053,Wiskitki,"[{'year': '2021', 'val': 0, 'attrId': 0}, {'ye...",199191,P2914
431356,71427338054,Wiskitki - miasto,"[{'year': '2021', 'val': 0, 'attrId': 0}, {'ye...",199191,P2914


In [19]:
# Collect all subject ids
subject_ids = {"BDL": [], "Census": {"1988" : [], "2002": [], "2011": [], "2021": []}}

subject_ids["BDL"] = list(df_demographic['subjectId'].unique())
subject_ids["Census"]["1988"] = list(df_c_1988['subjectId'].unique())
subject_ids["Census"]["2002"] = list(df_c_2002['subjectId'].unique())
subject_ids["Census"]["2011"] = list(df_c_2011['subjectId'].unique())
subject_ids["Census"]["2021"] = list(df_c_2021['subjectId'].unique())

subject_ids_flat = subject_ids['BDL'] + subject_ids["Census"]["1988"] + subject_ids["Census"]["2002"] \
    + subject_ids["Census"]["2011"] + subject_ids["Census"]["2021"]

subject_names_dict = {}
for subject in subject_ids_flat:
    subject_names_dict[subject] = ''
    
subject_ids

{'BDL': ['P1336', 'P2137', 'P2914'],
 'Census': {'1988': ['P2884', 'P2885', 'P2883', 'P2887'],
  '2002': ['P2114', 'P2403', 'P2402', 'P2871'],
  '2011': ['P3304', 'P3311', 'P3309', 'P3310', 'P3420'],
  '2021': ['P4253', 'P4320', 'P4345', 'P4287']}}

In [20]:
# Names of different subjects

subject_names_dict['P1336'] = 'pop__sex_URsplit'
subject_names_dict['P2137'] = 'pop__age_sex'
subject_names_dict['P2914'] = 'pop__sex_cities'

subject_names_dict['P2884'] = 'pop__age'
subject_names_dict['P2885'] = 'pop__educ'
subject_names_dict['P2883'] = 'pop__sex'
subject_names_dict['P2887'] = 'hh_size'

subject_names_dict['P2114'] = 'pop__age_sex'
subject_names_dict['P2403'] = 'pop__age_educ'
subject_names_dict['P2402'] = 'pop__sex_educ'
subject_names_dict['P2871'] = 'hh_size'

subject_names_dict['P3304'] = 'pop__age_sex'
subject_names_dict['P3311'] = 'pop__age_educ'
subject_names_dict['P3309'] = 'pop__sex_educ'
subject_names_dict['P3310'] = 'pop__educ_URsplit'
subject_names_dict['P3420'] = 'hh_size'

subject_names_dict['P4253'] = 'pop__age_sex'
subject_names_dict['P4320'] = 'pop__age_educ'
subject_names_dict['P4345'] = 'pop__sex_educ_URsplit'
subject_names_dict['P4287'] = 'hh_size'

subject_names_dict

{'P1336': 'pop__sex_URsplit',
 'P2137': 'pop__age_sex',
 'P2914': 'pop__sex_cities',
 'P2884': 'pop__age',
 'P2885': 'pop__educ',
 'P2883': 'pop__sex',
 'P2887': 'hh_size',
 'P2114': 'pop__age_sex',
 'P2403': 'pop__age_educ',
 'P2402': 'pop__sex_educ',
 'P2871': 'hh_size',
 'P3304': 'pop__age_sex',
 'P3311': 'pop__age_educ',
 'P3309': 'pop__sex_educ',
 'P3310': 'pop__educ_URsplit',
 'P3420': 'hh_size',
 'P4253': 'pop__age_sex',
 'P4320': 'pop__age_educ',
 'P4345': 'pop__sex_educ_URsplit',
 'P4287': 'hh_size'}

In [7]:
# =============================================================================
# STEP 4: Process Subjects
# =============================================================================
# Use the new process_subject_data() static method on GeoTERYTDatabase

# Example: Process subject P2137 (Population Data) from BDL
subject_id = 'P2137'
df_p2137 = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, subject_id)

print(f"Processed P2137: {df_p2137.shape}")
print(f"\nColumns: {list(df_p2137.columns)}")
print(f"\nCategory columns present:")
for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
    if col in df_p2137.columns:
        print(f"  {col}: {sorted(df_p2137[col].dropna().unique())}")
print(f"\nYears: {sorted(df_p2137['year'].dropna().astype(str).unique())}")
print(f"Unique TERYT IDs: {df_p2137['teryt_id'].nunique()}")
df_p2137.head()

Processed P2137: (7369080, 11)

Columns: ['nuts_id', 'name', 'variableId', 'subjectId', 'var_id', 'n1', 'n2', 'year', 'val', 'attrId', 'teryt_id']

Category columns present:
  n1: ['0-14', '0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70 i więcej', '70-74', '75-79', '80-84', '85 i więcej', 'ogółem']
  n2: ['kobiety', 'mężczyźni', 'ogółem']

Years: ['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Unique TERYT IDs: 4556


,nuts_id,name,variableId,subjectId,var_id,n1,n2,year,val,attrId,teryt_id
0,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1995,38609399.0,1.0,0000000
1,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1996,38639341.0,1.0,0000000
2,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1997,38659979.0,1.0,0000000
3,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1998,38666983.0,1.0,0000000
4,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1999,38263303.0,1.0,0000000


In [ ]:
# Example: Process subject P2114 (Population Data) from Census data 2002

In [8]:
# All subjects

df_subjects = {
    "BDL": df_demographic,
    "Census": {
        "1988": df_c_1988,
        "2002": df_c_2002,
        "2011": df_c_2011,
        "2021": df_c_2021
    }
}

df_processed_subjects = {
    "BDL": {},
    "Census": {
        "1988": {},
        "2002": {},
        "2011": {},
        "2021": {}
    }
}

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Processing BDL subject: {s}...")
            df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
            df_processed_subjects["BDL"][s] = df
    else:
        for sub in subject[1].items():
            print(f"Processing Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            df_c = df_subjects["Census"][year]
            for s in sub[1]:
                print(f"  Processing subject: {s}...")
                df = gtdb.GeoTERYTDatabase.process_subject_data(df_c, df_c_variables, s)
                df_processed_subjects["Census"][year][s] = df

del df_subjects
gc.collect()

# Save df_processed_subjects for later use
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'wb') as f:
    pickle.dump(df_processed_subjects, f)

Processing BDL subject: P1336...
Processing BDL subject: P2137...
Processing BDL subject: P2914...
Processing Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Processing subject: P2884...
  Processing subject: P2885...
  Processing subject: P2883...
  Processing subject: P2887...
Processing Census subject: 2002 - ['P2114', 'P2403', 'P2402', 'P2871']...
  Processing subject: P2114...
  Processing subject: P2403...
  Processing subject: P2402...
  Processing subject: P2871...
Processing Census subject: 2011 - ['P3304', 'P3311', 'P3309', 'P3310', 'P3420']...
  Processing subject: P3304...
  Processing subject: P3311...
  Processing subject: P3309...
  Processing subject: P3310...
  Processing subject: P3420...
Processing Census subject: 2021 - ['P4253', 'P4320', 'P4345', 'P4287']...
  Processing subject: P4253...
  Processing subject: P4320...
  Processing subject: P4345...
  Processing subject: P4287...


In [24]:
# Open the saved processed data to verify
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'rb') as f:
    df_processed_subjects = pickle.load(f)
    

In [30]:
df_c_1988

,id,name,values,variableId,subjectId
0,11212001011,Bochnia,"[{'attrId': 1, 'val': 28379, 'year': '1988'}]",196133,P2884
1,11212001022,Bochnia,"[{'attrId': 1, 'val': 15384, 'year': '1988'}]",196133,P2884
2,11212001032,Drwinia,"[{'attrId': 1, 'val': 6099, 'year': '1988'}]",196133,P2884
3,11212001042,Lipnica Murowana,"[{'attrId': 1, 'val': 5017, 'year': '1988'}]",196133,P2884
4,11212001052,Łapanów,"[{'attrId': 1, 'val': 6819, 'year': '1988'}]",196133,P2884
...,...,...,...,...,...
68946,71427338024,Mszczonów - miasto,"[{'attrId': 1, 'val': 959, 'year': '1988'}]",196152,P2887
68947,71427338025,Mszczonów - obszar wiejski,"[{'attrId': 1, 'val': 531, 'year': '1988'}]",196152,P2887
68948,71427338032,Puszcza Mariańska,"[{'attrId': 1, 'val': 1071, 'year': '1988'}]",196152,P2887
68949,71427338042,Radziejowice,"[{'attrId': 1, 'val': 579, 'year': '1988'}]",196152,P2887


In [31]:
df_processed_subjects["BDL"]["P2137"]

,nuts_id,name,variableId,subjectId,var_id,n1,n2,year,val,attrId,teryt_id
0,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1995,38609399.0,1.0,0000000
1,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1996,38639341.0,1.0,0000000
2,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1997,38659979.0,1.0,0000000
3,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1998,38666983.0,1.0,0000000
4,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1999,38263303.0,1.0,0000000
...,...,...,...,...,...,...,...,...,...,...,...
7369075,071427338054,Wiskitki - miasto,454046,P2137,454046,0-14,kobiety,2024,123.0,1.0,1438054
7369076,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2021,701.0,1.0,1438055
7369077,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2022,704.0,1.0,1438055
7369078,071427338055,Wiskitki - obszar wiejski,454046,P2137,454046,0-14,kobiety,2023,686.0,1.0,1438055


In [29]:
df_processed_subjects['Census']['1988']['P2883']

,nuts_id,name,variableId,subjectId,var_id,n1,teryt_id
0,011212001011,Bochnia,196130,P2883,196130,ogółem,1201011
1,011212001011,Bochnia,196130,P2883,196130,ogółem,1201011
2,011212001011,Bochnia,196130,P2883,196130,ogółem,1201011
3,011212001022,Bochnia,196130,P2883,196130,ogółem,1201022
4,011212001022,Bochnia,196130,P2883,196130,ogółem,1201022
...,...,...,...,...,...,...,...
32656,071427338042,Radziejowice,196132,P2883,196132,kobiety,1438042
32657,071427338042,Radziejowice,196132,P2883,196132,kobiety,1438042
32658,071427338052,Wiskitki,196132,P2883,196132,kobiety,1438052
32659,071427338052,Wiskitki,196132,P2883,196132,kobiety,1438052


In [11]:
# =============================================================================
# STEP 5: Load Subject Data onto TERYTRecords
# =============================================================================
# This attaches time series data to each matching TERYTRecord in the database

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Loading BDL subject: {s}...")
            df = df_processed_subjects["BDL"][s]
            stats = db.load_subject_data(df, source_type='BDL', subject_id=s, subject_name=subject_names_dict[s])
            print(f"  Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
            print(f"  Total data points loaded: {stats['total_data_points']:,}")
    else:
        for sub in subject[1].items():
            print(f"Loading Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            for s in sub[1]:
                print(f"  Loading subject: {s}...")
                df = df_processed_subjects["Census"][year][s]
                stats = db.load_subject_data(df, source_type='Census', subject_id=s, subject_name=subject_names_dict[s])
                print(f"    Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
                print(f"    Total data points loaded: {stats['total_data_points']:,}")

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

# Free memory
del df_p2137
del df_processed_subjects
gc.collect()

Loading BDL subject: P1336...
  ✓ Loaded 2,300,886 data points for subject P1336
  ✓ Matched 4569 TERYT records, 10 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1207132', '1210001', '1431981', '1431991', '1465158', '1465998', '2002162']
  Loading statistics: 4569 matched, 10 unmatched
  Total data points loaded: 2,300,886
Loading BDL subject: P2137...
  ✓ Loaded 7,351,704 data points for subject P2137
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4548 matched, 8 unmatched
  Total data points loaded: 7,351,704
Loading BDL subject: P2914...
  ✓ Loaded 1,508,400 data points for subject P2914
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4548 matched, 8 unmatched
  Tota

0

In [12]:
# =============================================================================
# STEP 6: Verify Data on Individual Records
# =============================================================================
# Check data summary
data_summary = db.get_data_summary()
print("Data Summary:")
for k, v in data_summary.items():
    print(f"  {k}: {v}")

# Look at a specific record (e.g., Kraków)
print("\n--- Example: Kraków ---")
krakow_records = [r for r in db._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow_records:
    rec = krakow_records[0]
    print(f"Record: {rec}")
    print(f"Data keys: {rec.list_data_keys()[:5]}...")
    print(f"Total data series: {rec.n_data_series}")
    
    # Show one time series
    if rec.data:
        first_key = list(rec.data.keys())[0]
        series = rec.data[first_key]
        print(f"\nFirst series: {series}")
        print(f"  Years: {series.years[:5]}...{series.years[-3:]}")
        print(f"  Value in 2020: {series.get_value(2020)}")
        print(f"  Categories: {series.categories}")

Data Summary:
  records_with_data: 4532
  total_records: 4560
  subjects: ['P1336', 'P2137', 'P2914']
  n_subjects: 3
  total_data_series: 420213
  total_data_points: 10893627

--- Example: Kraków ---
Record: TERYTRecord(1261011, Kraków, years=1999-2024, no changes)
Data keys: [('BDL', 'P1336', '60606'), ('BDL', 'P1336', '60607'), ('BDL', 'P1336', '60609'), ('BDL', 'P1336', '60611'), ('BDL', 'P1336', '60614')]...
Total data series: 93

First series: DataSeries(BDL/P1336/60606, cats=[n1=ogółem, n3=stan na 31 grudnia, n4=kobiety], 30 years [1995-2024])
  Years: [1995, 1996, 1997, 1998, 1999]...[2022, 2023, 2024]
  Value in 2020: 426639.0
  Categories: {'n1': 'ogółem', 'n3': 'stan na 31 grudnia', 'n4': 'kobiety'}


In [ ]:
# =============================================================================
# STEP 7: Aggregate Data for a Voivodeship (Regional Totals)
# =============================================================================
# Get all gminas in Małopolskie (voivodeship code '12') for year 2020
malopolskie_gminas = db.get_gminas_in_voivodeship('12', year=2020)
print(f"Małopolskie gminas in 2020: {len(malopolskie_gminas)}")

# Aggregate all population variables for the voivodeship
agg_df = db.aggregate_data(malopolskie_gminas, 'P2137', 2020, agg_func='sum')
print(f"\nAggregated data shape: {agg_df.shape}")
print(f"Columns: {list(agg_df.columns)}")
agg_df

In [ ]:
# =============================================================================
# STEP 8: Joint and Marginal Distributions
# =============================================================================
# Joint distribution: age group (n1) × gender (n2) for Małopolskie, year 2020
print("=== Joint Distribution (age group × gender) - Małopolskie 2020 ===")
joint = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                            row_category='n1', col_category='n2')
display(joint)

# Marginal distribution: by gender only
print("\n=== Marginal Distribution (by gender) - Małopolskie 2020 ===")
marginal_gender = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                       col_category='n2')
display(marginal_gender)

# Marginal distribution: by age group only
print("\n=== Marginal Distribution (by age group) - Małopolskie 2020 ===")
marginal_age = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                    row_category='n1')
display(marginal_age)

In [2]:
# =============================================================================
# STEP 9: Save Database with Data and Verify Persistence
# =============================================================================
# Save the database with data attached
# save_path = geo_root / 'geoteryt_complete_final.pkl'
# db.save_complete(save_path)

# Reload and verify
db = gtdb.load_complete_database(geo_root / 'geoteryt_complete_final.pkl')
db.print_summary()

# Verify data survived the save/load cycle
summary2 = db.get_data_summary()
print(f"\nData after reload: {summary2['records_with_data']} records with data, "
      f"{summary2['total_data_points']:,} total points")

# Quick sanity check: same Kraków record
krakow2 = [r for r in db._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow2:
    print(f"\nKraków data series after reload: {krakow2[0].n_data_series}")
    first_key = list(krakow2[0].data.keys())[0]
    print(f"First series: {krakow2[0].data[first_key]}")

gc.collect()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  Database version: 4.0
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
  ✓ Records with data: 4532
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geomet

0

In [14]:
db.get_by_teryt_id('1201011').get_data_by_subject('P2137')

{('BDL',
  'P2137',
  '47693'): DataSeries(BDL/P2137/47693, cats=[n1=60-64, n2=kobiety], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47694'): DataSeries(BDL/P2137/47694, cats=[n1=25-29, n2=ogółem], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47695'): DataSeries(BDL/P2137/47695, cats=[n1=30-34, n2=kobiety], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47696'): DataSeries(BDL/P2137/47696, cats=[n1=25-29, n2=kobiety], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47698'): DataSeries(BDL/P2137/47698, cats=[n1=40-44, n2=kobiety], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47701'): DataSeries(BDL/P2137/47701, cats=[n1=35-39, n2=ogółem], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47702'): DataSeries(BDL/P2137/47702, cats=[n1=55-59, n2=kobiety], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47706'): DataSeries(BDL/P2137/47706, cats=[n1=50-54, n2=mężczyźni], 30 years [1995-2024]),
 ('BDL',
  'P2137',
  '47707'): DataSeries(BDL/P2137/47707, cats=[n1=40-44, n2=ogółem], 30 years [1995-2